# Credit Card Default Risk Modeling

Predict whether a credit-card client will default next month using the UCI Taiwan credit-card dataset (30,000 clients).

This notebook is the **enhanced** analysis for the project. It keeps the exploratory spirit of the original tree-based study and upgrades the modeling stack to:

- stratified train/test splits and class-imbalance-aware estimators
- repayment-stress feature engineering (utilization, payment ratios, delay aggregates)
- modern baselines and boosting models (Logistic Regression, Random Forest, Extra Trees, HistGradientBoosting, XGBoost, LightGBM)
- ranking metrics (ROC-AUC, PR-AUC) plus F1-optimized decision thresholds

For a reproducible CLI run, use `python scripts/train_models.py` from the repository root.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay, confusion_matrix
from sklearn.model_selection import train_test_split

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from credit_default.data import basic_clean, load_raw
from credit_default.evaluate import classification_metrics, metrics_table, predict_proba_positive, tune_threshold
from credit_default.features import engineer_features, split_xy
from credit_default.models import fit_pipeline, get_model_candidates, make_pipeline
from credit_default.train import compare_models

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 1738
print("Project root:", ROOT)

## 1. Load and inspect the data

The dataset contains demographics, credit limit, six months of repayment status, bill amounts, and payment amounts, plus the binary target `default payment next month`.

In [ ]:
raw = basic_clean(load_raw())
print("Shape:", raw.shape)
print("Missing values:", int(raw.isna().sum().sum()))
display(raw.head())
raw["default"].value_counts(normalize=True).rename("share").to_frame()

## 2. Exploratory view of repayment risk

Recent repayment delay (`PAY_1`) is one of the strongest raw signals. Defaults concentrate among clients who are already behind.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(data=raw, x="default", ax=axes[0], palette="muted")
axes[0].set_title("Default rate (~22% positive class)")
axes[0].set_xlabel("Default next month")

pay_default = raw.groupby("PAY_1")["default"].mean().reset_index()
sns.barplot(data=pay_default, x="PAY_1", y="default", ax=axes[1], color="#2a6f97")
axes[1].set_title("Default rate by most recent repayment status")
axes[1].set_ylabel("Default rate")

plt.tight_layout()
plt.show()

## 3. Feature engineering

Beyond the original columns, we build leakage-safe features that summarize credit stress:

- mean / max / std of repayment delays
- credit utilization over the last six months
- payment-to-bill ratios
- counts of delayed and zero-payment months
- a composite stress score

In [ ]:
X, y = split_xy(raw)
print("Feature matrix:", X.shape)
engineered_preview = [
    "PAY_MEAN", "PAY_MAX", "N_DELAYED", "UTIL_MEAN", "UTIL_TREND",
    "PAY_RATIO_MEAN", "ZERO_PAY_MONTHS", "STRESS_SCORE",
]
display(X[engineered_preview].describe().T.round(3))

## 4. Train / compare models

`compare_models` performs a stratified hold-out split, uses stratified CV probabilities to pick an F1-optimal threshold, then reports hold-out ranking and classification metrics.

In [ ]:
leaderboard, fitted, summary, splits = compare_models(X, y, test_size=0.25, cv_folds=3)
display(leaderboard.round(4))
print("Best model:", summary["best_model"])
print("Train size:", summary["n_train"], "| Test size:", summary["n_test"])

## 5. Inspect the winning model

ROC and precision-recall curves show ranking quality independent of a single cutoff. The confusion matrix uses the CV-tuned threshold so the operating point balances precision and recall for the minority default class.

In [ ]:
best_name = summary["best_model"]
best = fitted[best_name]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

# Refit display curves on the same hold-out definition used in compare_models.
y_prob = predict_proba_positive(best.pipeline, X_test)
y_pred = (y_prob >= best.best_threshold).astype(int)
metrics = classification_metrics(y_test, y_prob, threshold=best.best_threshold)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name=best_name)
axes[0].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], name=best_name)
axes[1].set_title("Precision-Recall curve")

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[2],
            xticklabels=["No default", "Default"],
            yticklabels=["No default", "Default"])
axes[2].set_title(f"Confusion matrix @ thr={best.best_threshold:.3f}")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Actual")
plt.tight_layout()
plt.show()

pd.Series(metrics).round(4).to_frame("value")

## 6. Quick single-model experiment

Use this cell to try one estimator interactively without rerunning the full leaderboard.

In [ ]:
name = "lightgbm"  # try: logistic_regression, random_forest, xgboost, lightgbm, ...
model = get_model_candidates(y_train.to_numpy())[name]
pipe = fit_pipeline(make_pipeline(model, X_train), X_train, y_train)
prob = predict_proba_positive(pipe, X_test)
thr = tune_threshold(y_test, prob, optimize="f1")
print(metrics_table({name: classification_metrics(y_test, prob, threshold=thr)}))

## 7. Calibrate, cost-tune, and explain the winner

After model selection, the winning pipeline is isotonic-calibrated, then given both an F1 threshold and a cost-sensitive threshold (`FP_cost=1`, `FN_cost=5` by default). SHAP ranks the features that drive default risk.

In [ ]:
from credit_default.train import finalize_best_model
from credit_default.explain import save_shap_artifacts

best = finalize_best_model(fitted, summary, splits, calibrate=True)
print("Cost threshold:", round(best.threshold_cost, 4))
print("F1 threshold:", round(best.threshold_f1, 4))
print("Calibration:", best.calibration)
display(pd.Series(best.metrics).round(4).to_frame("value"))

shap_paths = save_shap_artifacts(best.pipeline, splits[0], ROOT / "artifacts")
print(shap_paths)

## Takeaways

1. **Class imbalance matters**: optimizing raw accuracy or untuned recall can look strong while ranking poorly.
2. **Engineered repayment features** improve signal beyond raw PAY/BILL columns alone.
3. **Gradient boosting** typically leads tree bags on this dataset for ROC-AUC and PR-AUC.
4. **Calibration + cost-sensitive thresholds** turn a strong ranker into an actionable policy.
5. **SHAP** makes the score explainable to risk and collections stakeholders.

See `docs/DOCUMENTATION.md` for methodology details and `README.md` for project positioning and quickstart.